# HF Symptoms/Signs Concept Mapping

Semantic embedding matching against the full Condition/Observation/Measurement concept pool, ranked by cosine similarity with patient prevalence used as a narrow tiebreaker.
The goal was to identify concept IDs corresponding to these heart failure (HF) symptoms and signs for use in defining the HF cohort.

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
from typing import List, Dict
import torch
import torchaudio
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

## HF symptoms/signs term list

In [ ]:
hf_symptoms_signs = {
    # Symptoms of HF — Typical
    'breathlessness',
    'orthopnea',
    'paroxysmal nocturnal dyspnea',
    'reduced exercise tolerance',
    'fatigue',
    'ankle swelling',
    'inability to exercise',
    'swelling of parts of the body other than ankles',
    'bendopnea',

    # Symptoms of HF — Less typical
    'nocturnal cough',
    'wheezing',
    'bloated feeling',
    'postprandial satiety',
    'loss of appetite',
    'decline in cognitive function',
    'confusion',
    'depression',
    'dizziness',
    'syncope',

    # Signs of HF — More specific
    'elevated jugular venous pressure',
    'third heart sound',
    'summation gallop',
    'cardiomegaly',
    'laterally displaced apical impulse',
    'hepatojugular reflux',
    'cheyne stokes respiration',

    # Signs of HF — Less specific
    'peripheral edema',
    'ankle edema',
    'sacral edema',
    'scrotal edema',
    'pulmonary rales',
    'unintentional weight gain',
    'weight loss',
    'muscle wasting',
    'cachexia',
    'cardiac murmur',
    'pleural effusion',
    'tachycardia',
    'irregular pulse',
    'tachypnea',
    'hepatomegaly',
    'ascites',
    'cold extremities',
    'oliguria',
    'narrow pulse pressure',
}

term_list = list(hf_symptoms_signs)
print(f"total terms: {len(term_list)}")

## Embedding pipeline
Generate and save semantic embeddings for HF symptom/sign terms and standard concepts in the Condition, Observation, and Measurement domains. These embeddings will support concept matching to identify IDs for HF cohort definition.

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2') # all-MiniLM-L6-v2, a Sentence Transformer model with six Transformer layers that produces 384-dimensional sentence embeddings.

In [ ]:
candidate_pool_sql = """
    SELECT DISTINCT concept_id, concept_name, domain_id
    FROM `""" + os.environ["WORKSPACE_CDR"] + """.concept`
    WHERE standard_concept = 'S'
      AND domain_id IN ('Condition', 'Observation', 'Measurement')
"""
candidate_pool_df = pd.read_gbq(candidate_pool_sql, dialect="standard", progress_bar_type="tqdm_notebook")
print(f"candidate pool: {len(candidate_pool_df)} concepts")

In [ ]:
t0 = time.time()
concept_embeddings = model.encode(
    candidate_pool_df['concept_name'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"done in {(time.time()-t0)/60:.1f} min")

term_embeddings = model.encode(term_list, convert_to_numpy=True)

np.save('concept_embeddings.npy', concept_embeddings)
np.save('term_embeddings.npy', term_embeddings)
candidate_pool_df.to_csv('candidate_pool.csv', index=False)
with open('term_list.json', 'w') as f:
    json.dump(term_list, f)

### Load cached embeddings

In [ ]:
concept_embeddings = np.load('concept_embeddings.npy')
term_embeddings = np.load('term_embeddings.npy')
candidate_pool_df = pd.read_csv('candidate_pool.csv')
with open('term_list.json') as f:
    term_list = json.load(f)

print(f"concept_embeddings: {concept_embeddings.shape}")
print(f"term_embeddings: {term_embeddings.shape}")
print(f"candidate_pool_df: {len(candidate_pool_df)} rows")

## Top-K semantic candidates per term
Compute cosine similarity between HF symptom/sign terms and candidate concept names,then retain the 10 highest-scoring concepts per term for subsequent review and selection.

In [ ]:
similarity_matrix = cosine_similarity(term_embeddings, concept_embeddings)

top_k = 10
results = []
for i, term in enumerate(term_list):
    top_idx = np.argsort(similarity_matrix[i])[::-1][:top_k]
    for idx in top_idx:
        results.append({
            'search_term': term,
            'concept_id': candidate_pool_df.iloc[idx]['concept_id'],
            'concept_name': candidate_pool_df.iloc[idx]['concept_name'],
            'domain_id': candidate_pool_df.iloc[idx]['domain_id'],
            'similarity': similarity_matrix[i][idx]
        })

embedding_candidates_df = pd.DataFrame(results)

In [ ]:
embedding_candidates_df.to_csv('embedding_candidates_df.csv', index=False)

## Prevalence lookup + final ranking

**Ranking rule (per term):**
1. Drop any candidate with similarity below `MIN_SIMILARITY`.
2. Among the survivors, keep only those within `SIMILARITY_MARGIN` of the top similarity score for that term.
3. Among those near-tied contenders, prefer the one with the most real-world patients.

In [ ]:
candidate_ids = embedding_candidates_df['concept_id'].unique().tolist()
domain_table_map = {
    "Condition": ("condition_occurrence", "condition_concept_id"),
    "Observation": ("observation", "observation_concept_id"),
    "Measurement": ("measurement", "measurement_concept_id"),
}

prevalence_frames = []
for domain_name, (occurrence_table, concept_id_col) in domain_table_map.items():
    domain_ids = embedding_candidates_df.loc[
        embedding_candidates_df['domain_id'] == domain_name, 'concept_id'
    ].unique().tolist()

    if not domain_ids:
        continue

    print(f"[{domain_name}] counting prevalence for {len(domain_ids)} candidates...")
    t0 = time.time()

    id_str = ", ".join(str(c) for c in domain_ids)
    prevalence_sql = f"""
        SELECT
            {concept_id_col} AS concept_id,
            COUNT(DISTINCT person_id) AS n_patients
        FROM `{os.environ["WORKSPACE_CDR"]}.{occurrence_table}`
        WHERE {concept_id_col} IN ({id_str})
        GROUP BY {concept_id_col}
    """
    df = pd.read_gbq(prevalence_sql, dialect="standard")
    print(f"  done in {time.time()-t0:.1f}s — {len(df)} of {len(domain_ids)} have patients")
    prevalence_frames.append(df)

prevalence_df = pd.concat(prevalence_frames, ignore_index=True) if prevalence_frames else pd.DataFrame(columns=["concept_id", "n_patients"])

In [ ]:
SIMILARITY_MARGIN = 0.05  
MIN_SIMILARITY = 0.5      

merged = embedding_candidates_df.merge(prevalence_df, on='concept_id', how='left')
merged['n_patients'] = merged['n_patients'].fillna(0).astype(int)

final_rows = []
for term, group in merged.groupby('search_term'):
    group = group[group['similarity'] >= MIN_SIMILARITY]
    if group.empty:
        continue
    top_sim = group['similarity'].max()
    contenders = group[group['similarity'] >= top_sim - SIMILARITY_MARGIN]
    best_row = contenders.sort_values('n_patients', ascending=False).iloc[0]
    final_rows.append(best_row)

best_matches = pd.DataFrame(final_rows)
feature_concepts = dict(zip(best_matches['search_term'], best_matches['concept_id']))

print(f"embedding-ranked matches (similarity-first, prevalence tiebreak within {SIMILARITY_MARGIN}):")
for term, cid in sorted(feature_concepts.items()):
    row = best_matches[best_matches['concept_id'] == cid].iloc[0]
    print(f"  {term:30} -> {cid:10} ({row['concept_name']}, "
          f"domain={row['domain_id']}, sim={row['similarity']:.3f}, n={row['n_patients']})")

## Manual corrections 

**Overridden (embedding picked a bad candidate, replaced with a verified concept):**
- `breathlessness`: embedding selected "No breathlessness" (4060227, n=1) - Forced to `312437` (Dyspnea).
- `confusion`: embedding selected "Acute confusion" (4038794, n=15) — Overridden to the broader parent concept `381273` ("Confusional state", n=174) to capture more patients.
- `muscle wasting`: method only surfaced narrowly-qualified variants ("Pelvic muscle wasting", "...respiratory muscle wasting"), none with real prevalence. No clean, general "muscle wasting" concept exists in this CDR vocabulary. Merged into `134765` (Cachexia).
- `pulmonary rales`: embedding only found the overly generic "Pulmonary" (4139338, n=0). An earlier manual word-root search (`'%rales%'`) found a clean, correctly-named concept: `1990125` (Rales). Reused here.
- `postprandial satiety`: embedding only found the overly generic "Postprandial" (4088117, n=0). An earlier manual word-root search (`'early satiety'`) found `40481819` (Early satiety), the standard clinical synonym. Reused here.

**Dropped entirely (no usable concept found by any method):**
- `bendopnea`: best candidate was "Orthopnea" at similarity 0.616 — a distinct clinical entity, not an acceptable proxy. The term is recent (first described 2017) and likely has no dedicated SNOMED concept yet.
- `laterally displaced apical impulse`: best candidate was the bare word "Apical" (4184475, Observation, n=0) — too vague to be usable; this is a physical-exam palpation finding that does not appear to be independently coded in this CDR.
- `hepatomegaly`: no clean, unqualified concept exists in this CDR vocabulary under any search method — all candidates are qualified by an unrelated comorbidity/etiology (e.g. "...associated with AIDS", "...due to mononucleosis") or are semantically wrong ("Hepatoblastoma", "Hepatopleural"). 

In [ ]:
manual_corrections = {
    'breathlessness': 312437,        # Dyspnea — corrects polarity error ("No breathlessness")
    'confusion': 381273,             # Confusional state — broader parent over narrow "Acute confusion"
    'muscle wasting': 134765,        # merged into Cachexia — no clean concept exists
    'pulmonary rales': 1990125,      # Rales — clean concept found via manual word-root search
    'postprandial satiety': 40481819, # Early satiety — clean concept found via manual word-root search
}

feature_concepts.update(manual_corrections)

# Dropped entirely 
for dropped_term in ['bendopnea', 'laterally displaced apical impulse', 'hepatomegaly']:
    feature_concepts.pop(dropped_term, None)

print(f"final feature_concepts: {len(feature_concepts)} of {len(hf_symptoms_signs)} original terms")

In [ ]:
# Build enriched lookup

concept_ids = list(feature_concepts.values())
concept_id_str = ", ".join(str(cid) for cid in concept_ids)

domain_lookup_sql = """
    SELECT concept_id, concept_name, domain_id
    FROM `""" + os.environ["WORKSPACE_CDR"] + """.concept`
    WHERE concept_id IN (""" + concept_id_str + """)
"""
domain_lookup_df = pd.read_gbq(domain_lookup_sql, dialect="standard", progress_bar_type="tqdm_notebook")

domain_table_map = {
    "Condition": ("condition_occurrence", "condition_concept_id"),
    "Observation": ("observation", "observation_concept_id"),
    "Measurement": ("measurement", "measurement_concept_id"),
}

prevalence_frames = []
for domain_name, (occurrence_table, concept_id_col) in domain_table_map.items():
    domain_ids = domain_lookup_df.loc[
        domain_lookup_df['domain_id'] == domain_name, 'concept_id'
    ].unique().tolist()
    if not domain_ids:
        continue

    id_str = ", ".join(str(c) for c in domain_ids)
    sql = f"""
        SELECT {concept_id_col} AS concept_id, COUNT(DISTINCT person_id) AS n_patients
        FROM `{os.environ["WORKSPACE_CDR"]}.{occurrence_table}`
        WHERE {concept_id_col} IN ({id_str})
        GROUP BY {concept_id_col}
    """
    prevalence_frames.append(pd.read_gbq(sql, dialect="standard", progress_bar_type="tqdm_notebook"))

prevalence_df = pd.concat(prevalence_frames, ignore_index=True) if prevalence_frames else pd.DataFrame(columns=["concept_id", "n_patients"])

total_pop_sql = "SELECT COUNT(DISTINCT person_id) AS total_n FROM `" + os.environ["WORKSPACE_CDR"] + "`.person"
total_n = pd.read_gbq(total_pop_sql, dialect="standard")['total_n'].iloc[0]

# merge to lookup table
id_to_term = {v: k for k, v in feature_concepts.items()}
feature_concepts_full = domain_lookup_df.merge(prevalence_df, on='concept_id', how='left')
feature_concepts_full['n_patients'] = feature_concepts_full['n_patients'].fillna(0).astype(int)
feature_concepts_full['term'] = feature_concepts_full['concept_id'].map(id_to_term)
feature_concepts_full['total_n'] = total_n
feature_concepts_full['prevalence_pct'] = round(feature_concepts_full['n_patients'] / total_n * 100, 3)

feature_concepts_full = feature_concepts_full[['term', 'concept_id', 'concept_name', 'domain_id', 'n_patients', 'total_n', 'prevalence_pct']]
feature_concepts_full = feature_concepts_full.sort_values('term').reset_index(drop=True)
feature_concepts_full.to_csv('feature_concepts_full.csv', index=False)